# Diagnostics walkthrough

All four `bayesbreak.diagnostics` routines on one synthetic Bernoulli-logistic dataset.

In [ ]:
import numpy as np
from bayesbreak import (
    BayesBreakGaussian,
    BayesBreakLogisticNormal,
    run_dp_diagnostics,
    run_non_conjugate_diagnostics,
    run_prior_sensitivity,
    select_n_groups_by_holdout,
)

rng = np.random.default_rng(0)
n = 80
y_g = np.r_[rng.normal(0, 0.3, 30), rng.normal(2, 0.3, 30), rng.normal(-1, 0.3, 20)]
X = np.arange(n).reshape(-1, 1)

## 1. DP invariants (`run_dp_diagnostics`)
Checks `∑ P(k) = 1`, forward/backward agreement (`prop:fb-duality`),
`∑_i P(b_i | y, k_map) = k_map − 1`, and MAP backtrack consistency
(`thm:map-correctness`).

In [ ]:
est = BayesBreakGaussian(k_max=8).fit(X, y_g)
report = run_dp_diagnostics(est)
print(report.summary)
for c in report.checks:
    print(f'  - {c.name}: passed={c.passed}  detail={c.detail}')

## 2. Non-conjugate diagnostics (`run_non_conjugate_diagnostics`)
Measures the empirical `ε` of `ass:uniform-block-error` and the
worst-case TV bound `exp(2 k_max ε) − 1` from
`cor:probability-error-conversion`. Reference fit uses high-Q
Gauss–Hermite quadrature.

In [ ]:
theta = np.r_[np.full(20, -1.0), np.full(20, 1.0), np.full(20, -0.5)]
p = 1.0 / (1.0 + np.exp(-theta))
y_b = rng.binomial(1, p).astype(float)
X_b = np.arange(y_b.size).reshape(-1, 1)

ref = BayesBreakLogisticNormal(k_max=8, approx='quadrature', gh_points=80).fit(X_b, y_b)
lap = BayesBreakLogisticNormal(k_max=8, approx='laplace').fit(X_b, y_b)

diag = run_non_conjugate_diagnostics(lap, ref)
print('block_error_max:', diag.extra['block_error_max'])
print('pk_tv_empirical:', diag.extra['pk_tv_empirical'])
print('pk_tv_upper_bound:', diag.extra['pk_tv_upper_bound'])
print('theoretical_rate:', diag.extra['theoretical_rate'])
print('theoretical_rate_violated:', diag.extra['theoretical_rate_violated'])

## 3. Prior-sensitivity (`run_prior_sensitivity`)
Reruns the DP under perturbations of `p(k)` and the length factor
`g(ℓ)`; reports `Δ p(k|y)` and `Δ P(b_i|y, k_map)` per variant.
This is the §5b *partition-prior sensitivity* diagnostic.

In [ ]:
sens = run_prior_sensitivity(est)
for v in sens.extra['variants']:
    print(f"  {v['variant']}: Δ p(k|y) max={v['delta_pk_max']:.3f}, "
          f"TV={v['delta_pk_tv']:.3f}; Δ P(b|y) L1={v['delta_bm_l1']:.3f}")

## 4. Held-out G-selection (`select_n_groups_by_holdout`)
K-fold marginal log-likelihood over the sequence axis for the latent-
template mixture. Mitigates `rem:teicher-overspec` (overspecified-G
redundancy).

In [ ]:
# Two visibly different templates: smooth jump up vs sharp drop.
seqs_a = [np.r_[rng.normal(0, 0.2, 30), rng.normal(2, 0.2, 30)] for _ in range(6)]
seqs_b = [np.r_[rng.normal(0, 0.2, 15), rng.normal(-2, 0.2, 45)] for _ in range(6)]
sequences = seqs_a + seqs_b

sel = select_n_groups_by_holdout(
    BayesBreakGaussian(k_max=4), sequences, g_grid=(1, 2, 3), n_folds=3,
)
print('best_g =', sel.extra['best_g'])
for g, m, s in zip(sel.extra['g_grid'], sel.extra['mean_test_loglik'], sel.extra['std_test_loglik']):
    print(f'  G={g}: mean held-out log p(y) = {m:.2f}  (±{s:.2f})')